# AFS Parse

Runs the full extraction pipeline on one or more filings in `COMMON.PDF_STAGING`.

**Steps:**
1. Select a filing from `PDF_STAGING` (status = `pending`)
2. Provide org details if this is a new organization
3. Run `pipeline.process_filing()` — identify → classify → extract → load
4. On success, status is set to `done`; on failure, `failed`

In [ ]:
import sys, json
sys.path.insert(0, '../src')

from afs import pipeline

In [ ]:
# Show pending filings
import pandas as pd

df = session.sql("""
    SELECT STAGING_ID, FILENAME, TOTAL_PAGES, STATUS, EXTRACTED_AT
      FROM AUDITED_FINANCIALS.COMMON.PDF_STAGING
     WHERE STATUS IN ('pending', 'failed')
     ORDER BY EXTRACTED_AT
""").to_pandas()

print(f'{len(df)} filing(s) ready to parse:')
df

In [ ]:
STAGING_ID  = '749594dd-3376-4f4a-8011-150f7e83ff59'
ORG_HINT    = None
REPARSE     = True

In [ ]:
from snowflake.snowpark.functions import col

if STAGING_ID:
    staging_df = session.table('AUDITED_FINANCIALS.COMMON.PDF_STAGING').filter(
        col('STAGING_ID') == STAGING_ID
    )
else:
    staging_df = session.table('AUDITED_FINANCIALS.COMMON.PDF_STAGING').filter(
        col('STATUS').isin(['pending', 'failed'])
    )

rows = staging_df.select(
    col('STAGING_ID'),
    col('FILENAME'),
    col('FILING_ID'),
    col('TOTAL_PAGES'),
    col('PAGE_TEXTS').cast('STRING').alias('PAGE_TEXTS_STR'),
).order_by(col('EXTRACTED_AT')).collect()

print(f'Processing {len(rows)} filing(s)...')

In [ ]:
import logging
from snowflake.snowpark.functions import col, lit

logging.basicConfig(level=logging.INFO, format='%(levelname)s %(name)s %(message)s')

def _update_staging_status(session, staging_id, status):
    session.table('AUDITED_FINANCIALS.COMMON.PDF_STAGING').update(
        {'STATUS': status},
        col('STAGING_ID') == staging_id,
    )

for row in rows:
    staging_id  = row[0]
    filename    = row[1]
    filing_id   = row[2]
    total_pages = row[3]
    page_texts  = json.loads(row[4])

    print(f'\n=== {filename} ({total_pages} pages) ===')

    _update_staging_status(session, staging_id, 'processing')

    try:
        report = pipeline.process_filing(
            session,
            filename=filename,
            filing_id=filing_id,
            page_texts=page_texts,
            total_pages=total_pages,
            org_hint=ORG_HINT,
            reparse=REPARSE,
            staging_id=staging_id,
        )
        _update_staging_status(session, staging_id, 'done')
        print(json.dumps(report.get('stages', {}), indent=2, default=str))
        if report.get('skipped'):
            print(f'[skip] {report["skipped"]}')
        else:
            print('[ok]')
    except Exception as exc:
        _update_staging_status(session, staging_id, 'failed')
        print(f'[fail] {exc}')
        raise

In [ ]:
from snowflake.snowpark.context import get_active_session
session = get_active_session()

## Diagnosis: Investigate empty CORTEX.COMPLETE responses
Loads the failing filing's page texts, measures input size per note batch, and calls Cortex directly to inspect raw responses.

In [ ]:
import json
import sys
sys.path.insert(0, '../src')

from snowflake.cortex import Complete
from snowflake.snowpark.context import get_active_session
from afs import config as C
from afs.classify import classify_pages, group_by_label
from afs.identify import identify_filing
from afs.pipeline import _notes_grouped
from afs.pdf_ingest import get_page_texts

session = get_active_session()

FAILING_FILENAME = 'P11543622-P11192478-P11609998.pdf'

row = session.sql(f"""
    SELECT PAGE_TEXTS, TOTAL_PAGES
      FROM AUDITED_FINANCIALS.COMMON.PDF_STAGING
     WHERE FILENAME = '{FAILING_FILENAME}'
     LIMIT 1
""").collect()[0]

page_texts = json.loads(row[0]) if isinstance(row[0], str) else row[0]
total_pages = row[1]

print(f'Loaded {FAILING_FILENAME}: {total_pages} pages, {len(page_texts)} text entries')
print(f'Total text size: {sum(len(p.get("text", "")) for p in page_texts):,} chars')

In [ ]:
classifications = classify_pages(page_texts, total_pages)
ident = identify_filing(page_texts)
fy_labels = ident.years_shown
note_batches = _notes_grouped(classifications)

print(f'FY labels: {fy_labels}')
print(f'Note batches ({len(note_batches)}):')
for i, batch in enumerate(note_batches):
    text = get_page_texts(page_texts, pages=batch)
    print(f'  Batch {i}: pages {batch} -> {len(text):,} chars')

In [ ]:
# DIAGNOSIS COMPLETE — no need to burn Cortex credits calling the model.
#
# ROOT CAUSE: This filing was stored as a single-blob text entry (1 entry for 62 pages).
# get_page_texts() returns the ENTIRE 135K-char document for every note batch
# because it cannot split by page (see pdf_ingest.py lines 71-72).
#
# mistral-large2 context window is ~32K tokens. 135K chars ≈ 40-50K tokens,
# which overflows the context and causes empty responses.
#
# FIXES (pick one):
#   1. Re-ingest this PDF with per-page PARSE_DOCUMENT so page_texts has 62 entries
#   2. Add a fallback in get_page_texts() that splits single-blob text on
#      page-break markers (e.g. form-feed chars or heuristic page headers)
#   3. Truncate the text to the model's context window with a warning

print('Diagnosis: single-blob page_texts → 135K chars sent per batch → exceeds mistral-large2 context → empty response')
print(f'page_texts entries: {len(page_texts)}')
print(f'Total chars: {sum(len(p.get("text", "")) for p in page_texts):,}')
print(f'mistral-large2 context: ~32K tokens ≈ ~110K chars')
print(f'Each batch sends: 135,252 chars → OVERFLOW')

## Fix: Re-ingest failing filing with per-page text
The original `PARSE_DOCUMENT` call did not use `page_split: TRUE`, so all 62 pages were returned as a single blob (~135K chars). This overflows `mistral-large2`'s context window on every note extraction call.

The fix re-runs `PARSE_DOCUMENT` with `page_split: TRUE` and updates the `PDF_STAGING` row.

In [ ]:
%%sql -r reparse_result
SELECT SNOWFLAKE.CORTEX.PARSE_DOCUMENT(
    @AUDITED_FINANCIALS.COMMON.AFS_STAGE,
    'P11543622-P11192478-P11609998.pdf',
    {'mode': 'LAYOUT', 'page_split': TRUE}
) AS parsed_result

In [ ]:
import json
from snowflake.snowpark.context import get_active_session
session = get_active_session()

raw = reparse_result.iloc[0, 0]
parsed = json.loads(raw) if isinstance(raw, str) else raw

pages = parsed.get('pages', [])
page_texts_new = [
    {'page': int(p.get('index', i)) + 1, 'text': str(p.get('content', ''))}
    for i, p in enumerate(pages)
]

print(f'Pages extracted: {len(page_texts_new)}')
for p in page_texts_new[:3]:
    print(f"  Page {p['page']}: {len(p['text']):,} chars")
print(f'  ...')
for p in page_texts_new[-2:]:
    print(f"  Page {p['page']}: {len(p['text']):,} chars")
print(f'Total chars: {sum(len(p["text"]) for p in page_texts_new):,}')

In [ ]:
import hashlib

blob = json.dumps(page_texts_new, sort_keys=True, ensure_ascii=False).encode()
new_filing_id = hashlib.sha256(blob).hexdigest()
page_texts_json = json.dumps(page_texts_new, ensure_ascii=False)

session.sql("""
    UPDATE AUDITED_FINANCIALS.COMMON.PDF_STAGING
       SET PAGE_TEXTS = PARSE_JSON(?),
           TOTAL_PAGES = ?,
           FILING_ID = ?,
           STATUS = 'pending'
     WHERE FILENAME = 'P11543622-P11192478-P11609998.pdf'
""", params=[page_texts_json, len(page_texts_new), new_filing_id]).collect()

verify = session.sql("""
    SELECT FILENAME, TOTAL_PAGES, ARRAY_SIZE(PARSE_JSON(PAGE_TEXTS)) AS TEXT_ENTRIES, STATUS
      FROM AUDITED_FINANCIALS.COMMON.PDF_STAGING
     WHERE FILENAME = 'P11543622-P11192478-P11609998.pdf'
""").to_pandas()
print(verify.to_string(index=False))